# MNAI Nano 2 — 0.9B from scratch — Kaggle, step by step

Same training as the Colab notebook, but it runs on Kaggle's server
while your laptop is closed. One committed run (Save & Run All) has
a 12 hour cap, so each night trains one chunk and stops cleanly with
a checkpoint saved. Resume the next night from that checkpoint.

Before you run it, set these on the right, in Settings:
- Internet: ON (needed for the dataset stream and pip install)
- Accelerator: GPU T4 x2

Then use Save Version > Save & Run All. Do not run the cells one by
one and leave them, a committed run is what keeps going after you
close the tab. Your checkpoints save to /kaggle/working, which is
what the new version keeps. Each checkpoint is a few GB, and Kaggle
keeps 20 GB of output, so do not hoard old ones.


## Step 1 — Check the GPU

Run this. You should see a Tesla T4, 15GB or 16GB. If you see no GPU, change the runtime type first (see top).

In [ ]:
!nvidia-smi

## Step 2 — Clone the repo

This clones the public main repo. It is public, so no login or token is needed.

In [ ]:
!git clone https://github.com/MeowSamOPDZN/mnai-nano-2.git
%cd mnai-nano-2
!ls -lh

## Step 3 — Install everything

This installs torch, transformers, datasets, tokenizers and bitsandbytes. It takes a few minutes the first time.

In [ ]:
!pip install -q -r requirements.txt
!python -c "import torch; print('torch', torch.__version__, 'cuda?', torch.cuda.is_available())"

## Step 4 — Check the model shape

This proves the 0.9B model loads and runs. It should print 899,753,472 params (0.900B) and a logits shape ending in 32768. If this fails, do not go on to training, tell me the error.

In [ ]:
!python model.py

## Step 5 — Where the work saves (Kaggle)

No Drive here. Everything under /kaggle/working is kept as this
version's output, so checkpoints live at
/kaggle/working/out-09b/ckpt-100, ckpt-200, and so on. The cell
below makes that folder and pulls in a previous night's checkpoint
if you attached the last version's output as an input.


In [ ]:
!mkdir -p /kaggle/working/out-09b
import glob, os, shutil
found = sorted(glob.glob('/kaggle/input/**/ckpt-*', recursive=True))
for ck in found:
    name = os.path.basename(ck)
    dest = f'/kaggle/working/out-09b/{name}'
    if not os.path.exists(dest):
        print('restoring', ck, '->', dest)
        shutil.copytree(ck, dest)
tok_in = sorted(glob.glob('/kaggle/input/**/tokenizer.json', recursive=True))
if tok_in and not os.path.exists('tokenizer.json'):
    print('restoring tokenizer from', tok_in[0])
    shutil.copyfile(tok_in[0], 'tokenizer.json')
print('checkpoints in /kaggle/working/out-09b:', sorted(os.listdir('/kaggle/working/out-09b')))


## Step 6 — Train the tokenizer (once per run, skipped if it is already here)


In [ ]:
import os, subprocess
if os.path.exists('tokenizer.json'):
    print('tokenizer.json already here, skipping')
else:
    subprocess.run(['python', 'train_tokenizer.py', '--out', 'tokenizer.json',
                    '--num-docs', '50000'], check=True)
    print('tokenizer trained')
!cp tokenizer.json /kaggle/working/tokenizer.json
!ls -lh tokenizer.json


## Step 7 — Train the model (starts fresh, or resumes on its own)

One cell does both jobs. If Step 5 restored a checkpoint, it resumes from the newest one. If there is nothing to restore, it starts from scratch. You never have to change a checkpoint number.


In [ ]:
import glob, os, subprocess
ckpts = sorted(glob.glob('/kaggle/working/out-09b/ckpt-*'), key=lambda p: int(p.rsplit('-', 1)[-1]))
cmd = ['python', 'train.py', '--tokenizer', 'tokenizer.json',
       '--out', '/kaggle/working/out-09b', '--max-tokens', '1000000000',
       '--micro-batch', '4', '--grad-accum', '16',
       '--ckpt-every', '100', '--log-every', '1', '--max-hours', '10.5']
if ckpts:
    print('resuming from', ckpts[-1])
    cmd += ['--resume', ckpts[-1]]
else:
    print('no checkpoint found, starting fresh')
subprocess.run(cmd, check=True)


## Step 8 — See what saved so far


In [ ]:
!ls /kaggle/working/out-09b


## Step 9 — Test the model

Change ckpt-100 below to your newest checkpoint, then run it. It should complete the sentence. Early checkpoints will sound like nonsense, that is normal, it gets better with more tokens.

In [ ]:
!python generate.py --model /kaggle/working/out-09b/ckpt-100 --prompt "MNAI was created by" --max-new 120


## Step 10 — Make the GGUF

Change ckpt-100 below to the checkpoint you want to use. This clones llama.cpp, converts the checkpoint to GGUF, then quantizes it to Q4_K_M. The file is about 550MB and runs on a 4GB RAM laptop in LM Studio.

In [ ]:
!python convert_to_gguf.py --model /kaggle/working/out-09b/ckpt-100 --out mnai-nano-2-q4_k_m.gguf
!ls -lh *.gguf


## Step 11 — Download the GGUF

Run this, then download the file from the file browser on the left (folder icon) — right-click mnai-nano-2-q4_k_m.gguf > Download. Load it in LM Studio.

In [ ]:
!ls -lh mnai-nano-2-q4_k_m.gguf
print('done — download mnai-nano-2-q4_k_m.gguf from the file browser on the left')

## If something goes wrong

- **No GPU in Step 1:** Settings on the right > Accelerator > GPU T4 x2, then Save Version again. If Kaggle says no GPU is available, wait a bit and try again later.
- **Out of memory in Step 7:** lower micro-batch to 2: add --micro-batch 2 --grad-accum 32 to the train command.
- **Version did not save:** the run went past 12 hours. Lower --max-hours (try 10) and commit again.
- **Session ended:** start a new version, attach the last version's output as an input, run Step 5 to restore the checkpoint, then Step 7 — it resumes from the restored checkpoint on its own.
- **tokenizer.json missing:** run Step 6 again, it must exist before Step 7.
